# DB structure and basic queries

This notebook uses native sqlite3 to efficiently query the large Spotify databases.

In [17]:
import sqlite3
import pandas as pd
import random
from autoencoder.db import get_connection
from autoencoder.db import get_tables
from autoencoder.db import get_columns
from autoencoder.db import head_table
from autoencoder.db import query_to_df

## Connect to Databases

Create separate connections to each database. Connections are instant with native sqlite3.

In [8]:
conn  = get_connection()
print("Connected to database")

Connected to database


## Explore Schema

List tables and their columns in each database.

In [10]:
for table in get_tables(conn):
    columns = get_columns(conn, table)
    col_str = ", ".join(f"{name} ({dtype})" for name, dtype in columns)
    print(f"  {table}: {col_str}")

  album_images: album_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  albums: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), album_type (TEXT), available_markets_rowid (INTEGER), external_id_upc (TEXT), copyright_c (TEXT), copyright_p (TEXT), label (TEXT), popularity (INTEGER), release_date (TEXT), release_date_precision (TEXT), total_tracks (INTEGER), external_id_amgid (TEXT)
  artist_albums: artist_rowid (INTEGER), album_rowid (INTEGER), is_appears_on (INTEGER), is_implicit_appears_on (INTEGER), index_in_album (INTEGER)
  artist_genres: artist_rowid (INTEGER), genre (TEXT)
  artist_images: artist_rowid (INTEGER), width (INTEGER), height (INTEGER), url (TEXT)
  artists: rowid (INTEGER), id (TEXT), fetched_at (INTEGER), name (TEXT), followers_total (INTEGER), popularity (INTEGER)
  audio_features: track_rowid (INT), track_id (TEXT), time_signature (INT), tempo (INT), key (INT), mode (INT), danceability (REAL), energy (REAL), loudness (REAL), speechiness

## Sample Data

Fetch random samples from key tables.

In [15]:
def sample_table(
    conn: sqlite3.Connection,
    table: str,
    n: int = 1000,
) -> pd.DataFrame:
    """
    Get a random sample from a table using ROWID-based sampling.

    This is fast even on huge tables because it:
    1. Gets max ROWID (uses index, instant)
    2. Generates random ROWIDs in Python
    3. Fetches only those specific rows
    """
    # Get max rowid
    cursor = conn.execute(f"SELECT MAX(rowid) FROM {table}")
    max_rowid = cursor.fetchone()[0]

    if max_rowid is None or max_rowid == 0:
        return pd.DataFrame()

    # Generate random rowids (sample more than needed to account for gaps)
    sample_size = min(n * 3, max_rowid)
    random_rowids = random.sample(range(1, max_rowid + 1), sample_size)

    # Fetch rows with those rowids
    placeholders = ",".join(["?"] * len(random_rowids))
    query = f"SELECT * FROM {table} WHERE rowid IN ({placeholders}) LIMIT {n}"
    return pd.read_sql_query(query, conn, params=random_rowids)


In [18]:
# Sample tracks
tracks_sample = sample_table(conn, "tracks", n=5)
tracks_sample

,rowid,id,fetched_at,name,preview_url,album_rowid,track_number,external_id_isrc,popularity,available_markets_rowid,disc_number,duration_ms,explicit
0,4321093,45vnQjzepy3FbEqJwz6lHx,1744243200000,"Dardanus, RCT 35B, Acte IV Scène 5: Bruit de g...",https://p.scdn.co/mp3-preview/09f833dc37fd6b6c...,366822,20,FR50X1295160,0,3,2,97633,0
1,13690114,6epLejVEGihQ6p27cjBKoc,1744243200000,Un ballo in maschera*: Act I Scene 1: Posa in ...,https://p.scdn.co/mp3-preview/43352592b26fe4a9...,1639378,2,ITDQ70311852,0,3,1,186266,0
2,27759454,2MxBmjXsxXOyiA3ycVd2iC,1743033600000,"Fantasiestücke, Op. 12: I. Des Abends: Sehr in...",https://p.scdn.co/mp3-preview/b0c51f2dcdd01828...,2656690,1,CH1260660101,0,3,1,238453,0
3,40052463,7eOhpdwNhOFTuTGmHt7Rea,1744243200000,No Stoppin' Us,https://p.scdn.co/mp3-preview/2db05cf78415463a...,4669668,2,QZES51947153,0,3,1,147226,0
4,64040906,3Zye3XtD2GCtLrVshwK8MS,1743033600000,Gangslide,https://p.scdn.co/mp3-preview/e221133acff0a611...,9704303,2,QZTAW2293787,4,3,1,157022,1


In [20]:
# Sample artist-track connections
tracks_artists_sample = sample_table(conn, "track_artists", n=5)
tracks_artists_sample

,track_rowid,artist_rowid
0,1868890,1545189
1,24821882,6902176
2,31118027,5511244
3,33655901,5020012
4,62417120,4036685


In [22]:
# Sample audio features
audio_sample = sample_table(conn, "audio_features", n=5)
audio_sample

,track_rowid,track_id,time_signature,tempo,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,173991500,5PiCl1rPisibMohkIvvR82,3,126.447,1,1,0.700,0.183,-23.869,0.9390,0.926,0.000992,0.117,0.649
1,178230046,3FLRjcw9PzdCfotTiCBHrt,4,96.198,8,0,0.629,0.816,-7.381,0.1370,0.694,0.033400,0.119,0.328
2,162500848,6nJbZuT9rAM13yRvsFx2Gg,4,94.750,10,1,0.474,0.871,-4.493,0.1700,0.110,0.000000,0.332,0.532
3,195007081,7z4xO8dOk4SGCZ2h0zYFNu,4,115.975,11,1,0.695,0.364,-12.144,0.0471,0.859,0.843000,0.363,0.411
4,197679352,5c9EYiRIeR4uZJKbzulDBn,4,146.088,0,1,0.440,0.382,-17.885,0.0380,0.309,0.898000,0.126,0.446


In [24]:
# Sample artists
artists_sample = sample_table(conn, "artists", n=5)
artists_sample

,rowid,id,fetched_at,name,followers_total,popularity
0,250101,7ibKSoMh7KIN9FmXSDSX31,1743033600000,Feather_xx,42,2
1,2040151,5aiPwWvpWtHZIAkMgK3sEY,1743033600000,Stefano Dallaporta,10,1
2,2590264,4ycfQ9wt64pLhcBHwuznwO,1743033600000,Podravske Skitnice,0,0
3,3940231,3TSittjbJkJtvvNcgjfIqu,1743033600000,Diaz B feat Magnett,1,0
4,6233781,0v5qy5iI5OtzaooUp0sSMI,1743033600000,WhoSeanX,182,2


## Query Tracks with Audio Features

Since we have separate databases, we fetch track IDs first, then query audio features.

In [58]:
# Get some tracks
tracks = query_to_df(
    conn,
    "SELECT id, name, popularity, duration_ms FROM tracks LIMIT 5"
)

# Get audio features for those track IDs
track_ids_str = ", ".join(tracks["id"].map(lambda x: f"'{x}'"))
audio_features = query_to_df(
    conn,
    f"SELECT * FROM audio_features WHERE track_id IN ({track_ids_str})",
)

# Merge (audio uses track_id, tracks uses id)
tracks_with_features = tracks.merge(audio_features, left_on="id", right_on="track_id")
tracks_with_features

,id,name,popularity,duration_ms,track_rowid,track_id,time_signature,tempo,key,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,5xHgo5JN0wfsV41HnRaos5,The Giver,89,202768,1,5xHgo5JN0wfsV41HnRaos5,4,93.024,4,1,0.558,0.901,-3.208,0.0583,0.07310,0.000000,0.3480,0.678
1,4kcRyBdCtBxcq14yDzVjJ0,Another Life,43,139388,2,4kcRyBdCtBxcq14yDzVjJ0,4,150.013,4,1,0.630,0.887,-5.604,0.0718,0.00341,0.440000,0.2260,0.912
2,2x1TExAkrUFFxVk616CKw8,Lover Online,43,167444,3,2x1TExAkrUFFxVk616CKw8,4,131.961,11,1,0.768,0.844,-5.247,0.0949,0.09800,0.103000,0.1940,0.631
3,0Hf3fR6XKINmMB4Fey7TiH,Crash,56,147601,4,0Hf3fR6XKINmMB4Fey7TiH,4,132.078,8,1,0.740,0.866,-7.083,0.2730,0.51400,0.000035,0.2810,0.500
4,0nWMjJ0b226HZP139cPKqw,I Just Missed A Call,51,146424,5,0nWMjJ0b226HZP139cPKqw,4,133.022,2,1,0.693,0.727,-7.868,0.0911,0.55100,0.282000,0.0988,0.702
